<a href="https://colab.research.google.com/github/Shikher-jain/Hugging_Face_LLM/blob/main/Transformer/nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers==4.41.2 accelerate

In [2]:
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
import torch

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [3]:
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {device}")

Using device: 0


In [4]:
HF_TOKEN = userdata.get("HF_TOKEN")
login(HF_TOKEN)

In [5]:
# from getpass import getpass
# HF_TOKEN = getpass("Enter token: ")
# login(HF_TOKEN)

In [6]:
# -------------------------
# Sentiment
# -------------------------
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)

# -------------------------
# Zero-shot
# -------------------------
zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

# -------------------------
# Text Generation
# -------------------------
generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M",
    device=device
)

# -------------------------
# Fill-mask
# -------------------------
unmasker = pipeline("fill-mask", model="distilbert/distilroberta-base", device=device)

# -------------------------
# NER
# -------------------------
# ner = pipeline("ner", grouped_entities=True)
# ner = pipeline("ner",aggregation_strategy="simple")
ner = pipeline(
    "ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple"
)


# -------------------------
# QA
# -------------------------
qa = pipeline("question-answering")

# -------------------------
# Summarization
# -------------------------
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# -------------------------
# Translation
# -------------------------
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-fr-en")

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another a

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [7]:
print(sentiment_classifier("I've been waiting for a HuggingFace course my whole life."))
print(sentiment_classifier([
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!"
]))

[{'label': 'POSITIVE', 'score': 0.9598046541213989}]
[{'label': 'POSITIVE', 'score': 0.9598046541213989}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


In [8]:
text = "The new AI model outperforms previous benchmarks"
labels = ["sports", "technology", "politics"]

print(zero_shot_classifier(text, candidate_labels=labels))

print(zero_shot_classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"]
))

{'sequence': 'The new AI model outperforms previous benchmarks', 'labels': ['technology', 'sports', 'politics'], 'scores': [0.9906519651412964, 0.0070508066564798355, 0.0022972861770540476]}
{'sequence': 'This is a course about the Transformers library', 'labels': ['education', 'business', 'politics'], 'scores': [0.844595193862915, 0.11197695881128311, 0.04342786595225334]}


In [9]:
generator("In this course, we will teach you how to")

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1168: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


[{'generated_text': 'In this course, we will teach you how to use the Python programming language to solve real-world'}]

In [ ]:
unmasker("This course will teach you all about <mask> models.", top_k=2)

In [21]:
generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
    num_beams=2,
    do_sample=True,
    pad_token_id=generator.model.config.eos_token_id
)


[{'generated_text': 'In this course, we will teach you how to use the C++ programming language to build computer programs that solve practical problems.\n\nThe C++'},
 {'generated_text': 'In this course, we will teach you how to use the C++ programming language to build computer programs that solve practical problems.\n\nWe will start'}]

In [22]:
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

[{'entity_group': 'PER',
  'score': np.float32(0.9981694),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': np.float32(0.97960204),
  'word': 'Hugging Face',
  'start': 33,
  'end': 45},
 {'entity_group': 'LOC',
  'score': np.float32(0.9932106),
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

In [25]:
qa(
    question="Where do I work?",
    context="My name is Sylvain and I work at Hugging Face in Brooklyn",
)

{'score': 0.7549429535865784,
 'start': 33,
 'end': 46,
 'answer': 'Hugging jFace'}

In [24]:
summarizer(
    """
    America has changed dramatically during recent years. Not only has the number of
    graduates in traditional engineering disciplines such as mechanical, civil,
    electrical, chemical, and aeronautical engineering declined, but in most of
    the premier American universities engineering curricula now concentrate on
    and encourage largely the study of engineering science. As a result, there
    are declining offerings in engineering subjects dealing with infrastructure,
    the environment, and related issues, and greater concentration on high
    technology subjects, largely supporting increasingly complex scientific
    developments. While the latter is important, it should not be at the expense
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other
    industrial countries in Europe and Asia, continue to encourage and advance
    the teaching of engineering. Both China and India, respectively, graduate
    six and eight times as many traditional engineers as does the United States.
    Other industrial countries at minimum maintain their output, while America
    suffers an increasingly serious decline in the number of engineering graduates
    and a lack of well-educated engineers.
"""
)

[{'summary_text': ' America has changed dramatically during recent years . The number of engineering graduates in the U.S. has declined in traditional engineering disciplines such as mechanical, civil,    electrical, chemical, and aeronautical engineering . Rapidly developing economies such as China and India continue to encourage and advance the teaching of engineering .'}]

In [28]:
translator("Ce cours est produit par Hugging Face.")

[{'translation_text': 'This course is produced by Hugging Face.'}]

In [29]:
image_classifier = pipeline(
    task="image-classification", model="google/vit-base-patch16-224",device=device
)
result = image_classifier(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
)
print(result)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

[{'label': 'lynx, catamount', 'score': 0.43350017070770264}, {'label': 'cougar, puma, catamount, mountain lion, painter, panther, Felis concolor', 'score': 0.03479619696736336}, {'label': 'snow leopard, ounce, Panthera uncia', 'score': 0.03240185230970383}, {'label': 'Egyptian cat', 'score': 0.023944752290844917}, {'label': 'tiger cat', 'score': 0.022889167070388794}]


In [32]:
transcriber = pipeline(
task="automatic-speech-recognition", model="openai/whisper-large-v3",device=device)
result = transcriber(
    "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"
)
print(result)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.


{'text': ' I have a dream that one day this nation will rise up and live out the true meaning of its creed.'}
